## 1. Setup & Imports

In [ ]:
!apt-get update && apt-get install -y tesseract-ocr
!pip install python-docx presidio-analyzer presidio-anonymizer Faker pytesseract spacy spacy-transformers easyocr
!python -m spacy download en_core_web_trf

In [ ]:
import pandas as pd
from collections import Counter
import spacy
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider
from faker import Faker
import pytesseract
import docx
import easyocr
import io
from PIL import Image

## 2. Custom Regex & Presidio Analyzer Setup

In [ ]:
# Define NLP engine provider using spacy transformers
provider = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_trf"}]
})
nlp_engine = provider.create_engine()

# Initialize Presidio AnalyzerEngine
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["en"])

# 1. PAN Card Recognizer (Generalized standard)
pan_pattern = Pattern(name="pan_pattern", regex=r"\b[A-Z]{5}[0-9]{4}[A-Z]{1}\b", score=0.95)
pan_recognizer = PatternRecognizer(supported_entity="IN_PAN", patterns=[pan_pattern])
analyzer.registry.add_recognizer(pan_recognizer)

# 2. Aadhaar Card Recognizer (Generalized standard starts with 2-9)
aadhaar_pattern = Pattern(name="aadhaar_pattern", regex=r"\b[2-9]\d{3}\s\d{4}\s\d{4}\b", score=0.95)
aadhaar_recognizer = PatternRecognizer(supported_entity="IN_AADHAAR", patterns=[aadhaar_pattern])
analyzer.registry.add_recognizer(aadhaar_recognizer)

# 3. Date of Birth (DOB) Recognizer (Broad standards & constraints)
dob_regex = r"(?i)(?:DOB|Date of Birth|D\.O\.B\.|जन्म तिथि|जन्म की तारीख)\s*[:\-]?\s*([0-3]\d[/.\-.][0-1]\d[/.\-.](?:19|20)\d{2})"
dob_pattern = Pattern(name="dob_pattern", regex=dob_regex, score=0.95)
dob_recognizer = PatternRecognizer(supported_entity="DATE_OF_BIRTH", patterns=[dob_pattern])
analyzer.registry.add_recognizer(dob_recognizer)

# 4. Indian Phone Numbers Recognizer (Broad generalized formats)
phone_regex = r"(?:\+?\s*91[\-\s]?)?(?:\(?0\d{2,4}\)?[\-\s]*)?[6-9]\d{9}|\b0\d{2,4}[\-\s]?\d{6,8}\b"
phone_pattern = Pattern(name="indian_phone_pattern", regex=phone_regex, score=0.85)
phone_recognizer = PatternRecognizer(supported_entity="PHONE_NUMBER", patterns=[phone_pattern])
analyzer.registry.add_recognizer(phone_recognizer)

# 5. CIN (Corporate Identity Number) Recognizer
cin_pattern = Pattern(name="cin_pattern", regex=r"\b[LU]\d{5}[A-Z]{2}\d{4}[A-Z]{3}\d{6}\b", score=0.95)
cin_recognizer = PatternRecognizer(supported_entity="IN_CIN", patterns=[cin_pattern])
analyzer.registry.add_recognizer(cin_recognizer)

# 6. SEBI Registration Number Recognizer
sebi_pattern = Pattern(name="sebi_pattern", regex=r"\bIN[A-Z]{1,3}\d{7,9}\b", score=0.95)
sebi_recognizer = PatternRecognizer(supported_entity="IN_SEBI_REG", patterns=[sebi_pattern])
analyzer.registry.add_recognizer(sebi_recognizer)

# 7. Indian PIN Code Recognizer
pin_pattern = Pattern(name="pin_pattern", regex=r"\b[1-9][0-9]{5}\b", score=0.95)
pin_recognizer = PatternRecognizer(supported_entity="IN_PIN_CODE", patterns=[pin_pattern])
analyzer.registry.add_recognizer(pin_recognizer)

# 8. Slashed Contacts Recognizer (Flaw 3 Fix)
slashed_person_regex = r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)\s*/\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)"
slashed_person_pattern = Pattern(name="slashed_contacts", regex=slashed_person_regex, score=0.9)
slashed_person_recognizer = PatternRecognizer(supported_entity="PERSON", patterns=[slashed_person_pattern])
analyzer.registry.add_recognizer(slashed_person_recognizer)

# 9. URL/Domain Recognizer (Flaw 6 Fix)
url_regex = r"(?:www\.)?[a-zA-Z0-9-]+\.[a-z]{2,}(?:\.[a-z]{2})?"
url_pattern = Pattern(name="url_pattern", regex=url_regex, score=0.8)
url_recognizer = PatternRecognizer(supported_entity="URL", patterns=[url_pattern])
analyzer.registry.add_recognizer(url_recognizer)

# 10. Explicit Allowlist (Flaw 1 & 5 Fix)
ALLOWLIST = {
    "INDIA", "MAHARASHTRA", "MUMBAI", "PUNE", "DELHI",
    "SEBI", "BSE", "NSE", "ROC", "RBI", "ICICI", "HDFC",
    "COMPANIES ACT", "GOVERNMENT OF INDIA", "CENTRAL GOVERNMENT",
    "QUALIFIED INSTITUTIONAL BUYERS", "QIB", "QIBS",
    "NON-INSTITUTIONAL INVESTORS", "NII", "RETAIL INDIVIDUAL INVESTORS",
    "FINANCIAL EXPRESS", "JANSATTA", "LOKSATTA", "NEWSPAPER", "WIDELY CIRCULATED"
}

print("Presidio Analyzer & Generalized Recognizers initialized successfully!")\n

## 3. Faker Mapping Registry

In [ ]:
fake = Faker('en_US')
PII_REGISTRY = {}

def generate_matched_email(original_email: str) -> str:
    if '@' not in original_email:
        return fake.user_name() + "@" + fake.free_email_domain()
        
    local_part, _ = original_email.split('@', 1)
    local_part_lower = local_part.lower()
    
    matched = False
    for orig_val, fake_val in PII_REGISTRY.items():
        orig_parts = orig_val.lower().split()
        fake_parts = fake_val.lower().split()
        
        for i, o_part in enumerate(orig_parts):
            if len(o_part) > 2 and o_part in local_part_lower:
                f_part = fake_parts[i] if i < len(fake_parts) else fake_parts[0]
                local_part_lower = local_part_lower.replace(o_part, f_part)
                matched = True
                
    if not matched:
        return fake.user_name() + "@" + fake.free_email_domain()
        
    return local_part_lower + "@" + fake.free_email_domain()

def get_consistent_fake(original_value: str, entity_type: str) -> str:
    cleaned_value = original_value.strip()
    if cleaned_value in PII_REGISTRY:
        return PII_REGISTRY[cleaned_value]
        
    match entity_type:
        case 'PERSON':
            new_fake = fake.name()
        case 'ORGANIZATION':
            new_fake = fake.company()
        case 'LOCATION':
            new_fake = fake.address().replace('\n', ', ')
        case 'EMAIL_ADDRESS':
            new_fake = generate_matched_email(original_value)
        case 'PHONE_NUMBER':
            new_fake = fake.phone_number()
        case 'IN_PAN':
            new_fake = f"[PAN_REDACTED_{fake.bothify('####')}]"
        case 'IN_AADHAAR':
            new_fake = f"[AADHAAR_REDACTED_{fake.bothify('####')}]"
        case 'DATE_OF_BIRTH':
            new_fake = fake.date_of_birth(minimum_age=21, maximum_age=70).strftime("%d/%m/%Y")
        case 'IP_ADDRESS':
            new_fake = fake.ipv4()
        case 'IN_CIN':
            new_fake = f"[CIN_REDACTED_{fake.bothify('####')}]"
        case 'IN_SEBI_REG':
            new_fake = f"[SEBI_REG_REDACTED_{fake.bothify('####')}]"
        case 'IN_PIN_CODE':
            new_fake = fake.postcode()
        case _:
            new_fake = fake.word()
            
    PII_REGISTRY[cleaned_value] = new_fake
    return new_fake


## 4. Document & Image Parsing Engine

In [ ]:
import logging
import re
import io
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import docx

logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)

TARGET_ENTITIES = [
    'PERSON', 'ORGANIZATION', 'LOCATION', 'EMAIL_ADDRESS',
    'PHONE_NUMBER', 'IN_PAN', 'IN_AADHAAR', 'DATE_OF_BIRTH', 'IP_ADDRESS',
    'IN_CIN', 'IN_SEBI_REG', 'IN_PIN_CODE', 'URL'
]

def redact_text_block(text: str, analyzer):
    if not text or not text.strip():
        return text, []
        
    # Flaw 4 Fix: Sentence case instead of Title case to prevent capitalization of common nouns
    text_to_analyze = text.capitalize() if text.isupper() and len(text) > 20 else text
        
    results = analyzer.analyze(text=text_to_analyze, entities=TARGET_ENTITIES, language="en")
    
    valid_results = []
    for result in results:
        original_substring = text[result.start:result.end]
        
        # Flaw 1 & 2 Fix: Allowlist bypass
        if original_substring.upper().strip() in ALLOWLIST:
            continue
            
        if result.entity_type == "IN_SEBI_REG":
            if not original_substring.isupper():
                continue
                
        valid_results.append(result)
    
    # Fix Overlapping Spans
    valid_results = sorted(valid_results, key=lambda x: x.start)
    filtered_results = []
    for current in valid_results:
        if not filtered_results:
            filtered_results.append(current)
            continue
        previous = filtered_results[-1]
        if current.start < previous.end:
            if current.score > previous.score:
                filtered_results.pop()
                filtered_results.append(current)
        else:
            filtered_results.append(current)
            
    filtered_results = sorted(filtered_results, key=lambda x: x.start, reverse=True)
    
    redacted_text = text
    detected_entities = []
    for result in filtered_results:
        start, end = result.start, result.end
        original_substring = text[start:end]
        detected_entities.append((original_substring, result.entity_type))
        
        if result.entity_type == "URL":
            fake_substring = "[www.example-issuer.com]"
        else:
            fake_substring = get_consistent_fake(original_substring, result.entity_type)
            
        redacted_text = redacted_text[:start] + fake_substring + redacted_text[end:]
        
    return redacted_text, detected_entities

def process_image_ocr(image_bytes: bytes):
    try:
        image = Image.open(io.BytesIO(image_bytes))
        raw_text = pytesseract.image_to_string(image)
        
        # Flaw 8 Fix: Process entire string through standard block, avoiding brittle ad-hoc regex
        raw_text = re.sub(r'([a-z])([A-Z])', r'\1 \2', raw_text)
        
        return redact_text_block(raw_text, analyzer)
    except Exception:
        return "", []

def process_docx(input_path: str, output_path: str):
    doc = docx.Document(input_path)
    all_detections = []
    
    for p in doc.paragraphs:
        # Flaw 5 Fix: Removed SAFE_TERMS blanket bypass, paragraphs process cleanly
        redacted_text, detections = redact_text_block(p.text, analyzer)
        p.text = redacted_text
        all_detections.extend(detections)
            
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    redacted_text, detections = redact_text_block(p.text, analyzer)
                    p.text = redacted_text
                    all_detections.extend(detections)
                        
    for part in doc.part.related_parts.values():
        if part.content_type.startswith("image/"):
            image_bytes = part.blob
            redacted_ocr, detections = process_image_ocr(image_bytes)
            all_detections.extend(detections)
            
            try:
                # Flaw 7 Fix: Replace image inline with rendered synthetic text template
                img = Image.open(io.BytesIO(image_bytes))
                white_img = Image.new("RGB", img.size, (255, 255, 255))
                draw = ImageDraw.Draw(white_img)
                # Render synthesized text cleanly over the image box
                draw.text((10, 10), redacted_ocr[:1000], fill=(0, 0, 0)) 
                img_byte_arr = io.BytesIO()
                white_img.save(img_byte_arr, format=img.format or 'PNG')
                part._blob = img_byte_arr.getvalue()
            except Exception:
                pass
            
    # Remove Flaw 7 text dumping
    doc.save(output_path)
    return all_detections\n

## 5. Evaluation Metrics Calculation


In [ ]:
GROUND_TRUTH_BASELINE = {
    "PERSON": [
        "Kushal Subbayya Hegde", "Pushpa Kushal Hegde", "Rajesh Kushal Hegde", "Rohit Kushal Hegde",
        "VISHAL SINGH", "SUGRIV SINGH", "MERAJ KHAN", "Sudhdan Khan",
        "Kishan Rastogi", "Abhijit Diwan", "Lokesh Shah", "Soumavo Sarkar"
    ],
    "IN_PAN": ["NBWPS1951N"],
    "IN_AADHAAR": ["2943 6593 3461"],
    "DATE_OF_BIRTH": ["06/05/2000", "12/12/1988"],
    "LOCATION": ["saray dan shah, KATRAULI, Poore Durgi, Phoolpur, Allahabad, Uttar Pradesh, 212402"]
}

def score(predicted_entities: list, ground_truth: dict = GROUND_TRUTH_BASELINE) -> dict:
    '''Flaw 11 Fix: Uses token-level matching instead of dictionary key collisions'''
    results = {}
    total_tp = total_fp = total_fn = 0
    
    predicted_dict = {k: [] for k in ground_truth.keys()}
    for text, label in predicted_entities:
        if label in predicted_dict:
            predicted_dict[label].append(text)
            
    for key, truth_list in ground_truth.items():
        truth_set = set([t.strip().lower() for t in truth_list])
        pred_set = set([p.strip().lower() for p in predicted_dict[key]])
        
        tp = len(truth_set & pred_set)
        fp = len(pred_set - truth_set)
        fn = len(truth_set - pred_set)
        
        precision = tp / (tp + fp) if (tp + fp) else 1.0
        recall = tp / (tp + fn) if (tp + fn) else 1.0
        
        results[key] = {
            "true_positives": tp,
            "false_positives": fp,
            "false_negatives": fn,
            "precision": round(precision, 3),
            "recall": round(recall, 3),
        }
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
    overall_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 1.0
    overall_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 1.0
    
    results["overall"] = {
        "true_positives": total_tp,
        "false_positives": total_fp,
        "false_negatives": total_fn,
        "precision": round(overall_precision, 3),
        "recall": round(overall_recall, 3),
    }
    return results\n

## 6. Execution & Pipeline Evaluation


In [ ]:
import json
from collections import Counter

# Flaw 10 Fix: Evaluate across the full document text without arbitrary slicing
INPUT_FILE = "data/input/Red Herring Prospectus.docx"
OUTPUT_FILE = "data/output/Anonymized_Red_Herring_Prospectus.docx"

if __name__ == "__main__":
    try:
        print(f"Starting unified redaction pipeline on {INPUT_FILE}...")
        all_detections = process_docx(INPUT_FILE, OUTPUT_FILE)
        print(f"\nSuccess! Fully redacted document saved to: {OUTPUT_FILE}")
        
        counts = Counter([label for text, label in all_detections])
        print("\n=== REDACTION RUN STATS ===")
        print(f"Total Entities Redacted: {len(all_detections)}")
        for label, count in counts.most_common():
            print(f"  {label}: {count}")
                
        df_report = score(all_detections)
        print("\n=== FULL DOCUMENT EVALUATION ACCURACY ===")
        print(json.dumps(df_report, indent=2))
        
    except Exception as e:
        print(f"Execution Error: {e}")\n